# Trabajo Práctico 2 (TP2) - Parte 1: Preprocesamiento de Datos y Análisis Estadístico Exploratorio

### Machine Learning 1 (23433)
#### Facultad de Ingeniería - Universidad Nacional de Asunción (FIUNA)

---

## Objetivos de la Parte 1
1. Cargar e inspeccionar la estructura del conjunto de datos unificado de rendimiento académico de FIUNA (`reglamento_nuevo_unificado.csv`).
2. Mapear las 27 siglas e intensificaciones curriculares a las 7 carreras principales de la facultad.
3. Aplicar un preprocesamiento de integridad de datos **no destructivo** que preserve el 100% de los 64,295 registros sin eliminar filas.
4. Responder a 6 preguntas estadísticas exploratorias clave para auditar la masa estudiantil, la retención académica y las tasas de aprobación.


In [7]:
# 1. Carga de Librerías Fundamentales
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)

# Carga del Dataset
csv_path = 'reglamento_nuevo_unificado.csv'
if not os.path.exists(csv_path):
    csv_path = os.path.join('..', 'Clase_5', 'reglamento_nuevo_unificado.csv')

df_raw = pd.read_csv(csv_path)
print(f"Dataset cargado exitosamente: {df_raw.shape[0]:,} filas y {df_raw.shape[1]} columnas.")


Dataset cargado exitosamente: 64,295 filas y 29 columnas.


## Ejercicio 1: Mapeo de Intensificaciones Curriculares a Carreras Principales

Completa el diccionario `career_code_mapping` para mapear las 27 siglas del sistema (`CIV-PLS13`, `INT9CONSTR`, `ELE-PLS23`, `INT9SDIGYT`, `MCT-PLS13`, `IND-PLS13`, `CGF-PLS13`, `MEC-PLS13`, `ECA-PLS13`, etc.) a sus 7 carreras principales:
- `Ing. Civil`
- `Ing. Electrónica`
- `Ing. Mecatrónica`
- `Ing. Industrial`
- `Ing. Geográfica`
- `Ing. Mecánica`
- `Ing. Electromecánica`


In [8]:
# TODO: Completar la estructura del diccionario career_code_mapping
career_code_mapping = {
    # === ESCRIBE TU CÓDIGO AQUÍ ===
    'CIV-PLS13': 'Ing. Civil', 'CIV-PLS23': 'Ing. Civil', 'INT9CONSTR': 'Ing. Civil',
    'INT9TRANSP': 'Ing. Civil', 'INT9ORTERR': 'Ing. Civil', 'INT9SANEHI': 'Ing. Civil',
    
    'ELE-PLS13': 'Ing. Electrónica', 'ELE-PLS23': 'Ing. Electrónica',
    'INT9ELECTR': 'Ing. Electrónica', 'INT9SDIGYT': 'Ing. Electrónica',
    
    'MCT-PLS13': 'Ing. Mecatrónica', 'MCT-PLS23': 'Ing. Mecatrónica', 'MCT9-OPT': 'Ing. Mecatrónica',
    
    'IND-PLS13': 'Ing. Industrial', 'IND-PLS23': 'Ing. Industrial',
    'INT9G-ECO': 'Ing. Industrial', 'INT9-PROYT': 'Ing. Industrial',
    
    'CGF-PLS13': 'Ing. Geográfica', 'CGF-PLS23': 'Ing. Geográfica', 'INT9RNYMA': 'Ing. Geográfica',
    
    'MEC-PLS13': 'Ing. Mecánica', 'MEC-PLS23': 'Ing. Mecánica',
    'INT9MECANI': 'Ing. Mecánica', 'MEC9-OPT': 'Ing. Mecánica',
    
    'ECA-PLS13': 'Ing. Electromecánica', 'ECA-PLS23': 'Ing. Electromecánica', 'ECA9-OPT': 'Ing. Electromecánica'
}

df_clean = df_raw.copy()
# TODO: Asignar la nueva columna Carrera_Nombre usando map()
df_clean['Carrera_Nombre'] = df_clean['Cod.Car.Sec'].astype(str).str.strip().map(career_code_mapping)

# Definición del Target Binario (1 para 'S', 0 para 'N')
df_clean = df_clean.dropna(subset=['Aprobado', 'Carrera_Nombre']).copy()
df_clean['Target'] = (df_clean['Aprobado'] == 'S').astype(int)

print(f"Filas tras mapeo de carreras: {len(df_clean):,}")
# Verificación de las carreras
print("Carreras únicas en df_clean:")
print(df_clean['Carrera_Nombre'].unique())
# Mostrar numero de matriculaciones por carrera
print("\nMatriculaciones por carrera:")
print(df_clean['Carrera_Nombre'].value_counts())
# Verificacion de la suma de las matriculaciones
print("\nSuma de matriculaciones en df_clean")
print(df_clean['Carrera_Nombre'].value_counts().sum())
if (df_clean['Carrera_Nombre'].value_counts().sum() == len(df_raw)):
    print("\nVerificacion exitosa!")
else:
    print("\nVerificacion Fallida")

Filas tras mapeo de carreras: 64,295
Carreras únicas en df_clean:
<StringArray>
['Ing. Geográfica', 'Ing. Electrónica', 'Ing. Civil', 'Ing. Mecatrónica', 'Ing. Industrial', 'Ing. Electromecánica', 'Ing. Mecánica']
Length: 7, dtype: str

Matriculaciones por carrera:
Carrera_Nombre
Ing. Civil              29392
Ing. Electrónica        11928
Ing. Mecatrónica         7147
Ing. Industrial          6416
Ing. Geográfica          4626
Ing. Mecánica            3093
Ing. Electromecánica     1693
Name: count, dtype: int64

Suma de matriculaciones en df_clean
64295

Verificacion exitosa!


## Ejercicio 2: Limpieza Numérica y Generación de Atributos Derivados (Sin Eliminación de Filas)

Convierte los campos numéricos de parciales y evaluaciones a formato float utilizando `pd.to_numeric(..., errors='coerce')` para mantener el 100% de las filas.
Crea las siguientes variables derivadas:
- `Score_Parciales`: Promedio entre el 1er y 2do parcial.
- `Diff_Parciales`: Diferencia ($2^\circ\text{Par} - 1^\circ\text{Par}$).


In [9]:
# TODO: Limpiar y convertir atributos numéricos sin eliminar filas
# === ESCRIBE TU CÓDIGO AQUÍ ===
df_clean['Primer_Par_Clean'] = pd.to_numeric(df_clean['Primer.Par'], errors='coerce')
df_clean['Segundo_Par_Clean'] = pd.to_numeric(df_clean['Segundo.Par'], errors='coerce')
df_clean['Score_Parciales'] = (df_clean['Primer_Par_Clean'].fillna(0) + df_clean['Segundo_Par_Clean'].fillna(0)) / 2.0
df_clean['Diff_Parciales'] = df_clean['Segundo_Par_Clean'].fillna(0) - df_clean['Primer_Par_Clean'].fillna(0)
df_clean['TPLab_Clean'] = pd.to_numeric(df_clean['TPLab.'], errors='coerce')
df_clean['Asis_Clean'] = pd.to_numeric(df_clean['Asis'], errors='coerce')
df_clean['Firma_Clean'] = pd.to_numeric(df_clean['Firma'], errors='coerce')
df_clean['FirmaCalc_Clean'] = pd.to_numeric(df_clean['FirmaCalculada'], errors='coerce')

print(f"Filas preservadas en df_clean: {len(df_clean):,} (100% retención)")


Filas preservadas en df_clean: 64,295 (100% retención)


## Ejercicio 3: Auditoría Estadísticas Exploratoria (6 Preguntas Clave)

Responde a las siguientes 6 preguntas estadísticas utilizando código en Python sobre `df_raw` y `df_clean`:

1. **¿Cuántos estudiantes presentan registros en los tres ciclos del CSV?**
2. **¿Cuántas personas/registros tienen calificación final (`Nota.Final`)?**
3. **¿Cuántos tienen proceso (`FirmaCalculada` / `Firma`)?**
4. **¿Cuál es la tasa de aprobación global y por Carrera?**
5. **¿Cómo se comparan las notas medias del 1er y 2do Parcial según condición final (Aprobado vs No Aprobado)?**
6. **¿Cuál es la tasa de abandono / inasistencia total a parciales?**


In [45]:

# Pregunta 1) Cuantos estudiantes presentan registros en los tres ciclos del CSV
print("\n1) ---------")
print("\nEstudiantes con registros en los 3 ciclos: ")
numeroEstudiantes = df_raw.assign(Ciclo=df_raw["Anho"].astype(str) + df_raw["Semestre"].astype(str)).groupby("ALUMNO_ID")["Ciclo"].nunique().eq(3).sum()
print(numeroEstudiantes)
print(f"{numeroEstudiantes / df_raw['ALUMNO_ID'].nunique() * 100:.2f} %" )
# Pregunta 2) Cuantas personas/registros tienen calificacion final
print("\n2) ---------")
print("\nEstudiantes con calificacion final: ")
print((~df_raw["Nota.Final"].isna()).sum())
print("\nEstudiantes unicos:")
print(df_raw.loc[df_raw["Nota.Final"].notna() & df_raw["ALUMNO_ID"].notna(),"ALUMNO_ID"].nunique())
# Pregunta 3) Cuantas tienen proceso
print("\n3) ---------")


print("\nFirma Calculada: ")
print(df_raw["FirmaCalculada"].notna().sum())
print("\nFirma")
print((df_clean["Firma_Clean"] >= 50).sum())
# Pregunta 4) Cual es la tasa de aprobacion global y por carrera
print("\n4) ---------")
print("\nTasa de aprobación global:")

print(f"{df_clean['Target'].mean() * 100:.2f}%")

print("\nTasa de aprobación por carrera:")
tabla = df_clean.groupby("Carrera_Nombre").size().to_frame("Inscritos")
tabla["Aprobados"] = df_clean.groupby("Carrera_Nombre")["Target"].sum()
tabla["Tasa_Aprob_Pct"] = (tabla["Aprobados"] / tabla["Inscritos"] * 100).round(2)
pd.set_option("display.max_rows", None)
print(tabla)
# Pregunta 5) Como se comparan las notas medias del 1er y 2do Parcial segun condicion final
print("\n5) ---------")
print("\nPromedios de Parciales por Target:")
print(df_clean.groupby("Target")[["Primer_Par_Clean", "Segundo_Par_Clean"]].mean().round(2))
# Pregunta 6) Cual es la tasa de abandono / inasistencia total a parciales
print("\n6) ---------")
print(f"\nRegistros con 0 en ambos parciales: {( (df_clean['Primer_Par_Clean'].fillna(0) == 0) & (df_clean['Segundo_Par_Clean'].fillna(0) == 0) ).sum():,} ({(((df_clean['Primer_Par_Clean'].fillna(0) == 0) & (df_clean['Segundo_Par_Clean'].fillna(0) == 0)).sum() / len(df_clean) * 100):.2f}%)")


1) ---------

Estudiantes con registros en los 3 ciclos: 
3532
74.22 %

2) ---------

Estudiantes con calificacion final: 
42720

Estudiantes unicos:
4344

3) ---------

Firma Calculada: 
19407

Firma
41715

4) ---------

Tasa de aprobación global:
60.67%

Tasa de aprobación por carrera:
                      Inscritos  Aprobados  Tasa_Aprob_Pct
Carrera_Nombre                                            
Ing. Civil                29392      17202           58.53
Ing. Electromecánica       1693       1150           67.93
Ing. Electrónica          11928       6638           55.65
Ing. Geográfica            4626       3275           70.80
Ing. Industrial            6416       4050           63.12
Ing. Mecatrónica           7147       4691           65.64
Ing. Mecánica              3093       1999           64.63

5) ---------

Promedios de Parciales por Target:
        Primer_Par_Clean  Segundo_Par_Clean
Target                                     
0                  26.16              19.